# Multimodal Search Pipeline — AMI Meeting Corpus ES2008a

I built this pipeline to explore how the audio and visual content of a real meeting can be unified into a single semantic search system. The pipeline ingests a raw meeting recording (WAV) and the accompanying presentation slides (JPGs), transcribes and OCRs them locally, embeds everything into a shared vector space, and exposes a cross-modal search interface that lets you find relevant moments regardless of whether the answer is spoken or shown on a slide.

The sample dataset is meeting **ES2008a** from the [AMI Meeting Corpus](https://groups.inf.ed.ac.uk/ami/corpus/) — a 30-minute scenario meeting about product design, chosen because it is publicly available, compact enough to process in minutes on a laptop, and contains both spoken discussion and projected slides.

## Pipeline at a glance

```
WAV file  ──►  Whisper (ASR)  ──►  Transcript text
                                         │
Slide JPGs ──►  Tesseract (OCR) ──►  Slide text
                                         │
                      ┌──────────────────┘
                      ▼
             sentence-transformers          all-MiniLM-L6-v2
             (384-d embeddings)
                      │
           ┌──────────┴──────────┐
           ▼                     ▼
   ChromaDB collection    ChromaDB collection
   transcript_chunks        slides_ocr
           └──────────┬──────────┘
                      ▼
            Cross-modal semantic search
```

## Setup

I import every library up-front so missing dependencies surface immediately. The `step_times` dictionary collects wall-clock timings for the pipeline summary table in the Evaluation section.

In [2]:
import os
import sys
import glob
import time
import random
import platform
import shutil
import warnings
from pathlib import Path
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import whisper
import pytesseract
from PIL import Image
from sentence_transformers import SentenceTransformer
import chromadb
import jiwer

# Windows: point pytesseract at the default Tesseract install location.
# Adjust the path if you installed Tesseract elsewhere.
if platform.system() == "Windows":
    pytesseract.pytesseract.tesseract_cmd = (
        os.environ.get("TESSERACT_CMD",
                       r"C:\Program Files\Tesseract-OCR\tesseract.exe")
    )

# ── ffmpeg (required by Whisper for audio decoding) ────────────────────────
_ffmpeg_env = os.environ.get("FFMPEG_PATH")
if _ffmpeg_env:
    # Accept either the full path to the executable or its parent directory
    _ffmpeg_p = Path(_ffmpeg_env)
    _ffmpeg_dir = str(_ffmpeg_p.parent if _ffmpeg_p.suffix else _ffmpeg_p)
    os.environ["PATH"] = _ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

if shutil.which("ffmpeg") is None:
    print(
        "[WARNING] ffmpeg not found. Whisper transcription will fail.\n"
        "  Install ffmpeg and either add it to your system PATH or set\n"
        "  FFMPEG_PATH in .env (e.g. FFMPEG_PATH=C:\\ffmpeg\\bin\\ffmpeg.exe)"
    )
else:
    print(f"ffmpeg : {shutil.which('ffmpeg')}")

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = Path("data")
AUDIO_DIR      = DATA_DIR / "raw" / "audio"
SLIDES_DIR     = DATA_DIR / "raw" / "slides"
TRANSCRIPT_DIR = DATA_DIR / "raw" / "transcripts"
CHROMA_DIR     = Path("chroma_db")

for _d in [AUDIO_DIR, SLIDES_DIR, TRANSCRIPT_DIR, CHROMA_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
MEETING_ID    = "ES2008"
MEETING_PART  = "a"
CHUNK_SIZE    = 500
OVERLAP       = 50
WHISPER_MODEL = "base"
EMBED_MODEL   = "all-MiniLM-L6-v2"

# ── Timing registry ────────────────────────────────────────────────────────
step_times: dict = {}

print(f"Python {sys.version.split()[0]}  |  Platform: {platform.system()}")
print("Setup complete.")

ffmpeg : C:\Users\Zhuor\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-full_build\bin\ffmpeg.EXE
Python 3.12.7  |  Platform: Windows
Setup complete.


In [3]:
%pip install jiwer

Note: you may need to restart the kernel to use updated packages.


---
## Step 1 — Audio Transcription

I use OpenAI Whisper to transcribe the raw meeting recording. Whisper is an encoder-decoder transformer trained on 680k hours of multilingual speech; it runs fully offline once the model weights are downloaded. I default to `whisper-base` (74 M parameters) as a practical balance between speed and accuracy — the Evaluation section benchmarks it against `tiny` and `small`.

The output is stored in a tidy DataFrame with one row per meeting segment so it is easy to join against other metadata tables later.

In [4]:
audio_path = AUDIO_DIR / "ES2008a.Mix-Headset.wav"

if not audio_path.exists():
    raise FileNotFoundError(
        f"Audio file not found: {audio_path}\n"
        "Run  python download_data.py  to fetch the AMI corpus data."
    )

print(f"Loading Whisper model: {WHISPER_MODEL} …")
_t0 = time.time()
whisper_model = whisper.load_model(WHISPER_MODEL)

print(f"Transcribing {audio_path.name} …")
result         = whisper_model.transcribe(str(audio_path))
step_times["Transcription"] = time.time() - _t0

transcript_text = result["text"].strip()
audio_duration  = result.get("duration", 0.0)

transcription_df = pd.DataFrame([{
    "meeting_id"      : MEETING_ID,
    "meeting_part"    : MEETING_PART,
    "audio_file_path" : str(audio_path),
    "transcript_text" : transcript_text,
    "audio_duration"  : round(audio_duration, 2),
}])

print(f"\nDuration  : {audio_duration:.1f} s")
print(f"Characters: {len(transcript_text):,}")
print(f"Time      : {step_times['Transcription']:.1f} s")
print(f"\nFirst 100 chars of transcript:")
print(transcript_text[:100])
print()
display(transcription_df[["meeting_id", "meeting_part", "audio_duration"]])

Loading Whisper model: base …
Transcribing ES2008a.Mix-Headset.wav …

Duration  : 0.0 s
Characters: 11,761
Time      : 91.5 s

First 100 chars of transcript:
Okay, good morning everybody. I'm glad you could all come. I'm really excited to start this team. I'



,meeting_id,meeting_part,audio_duration
0,ES2008,a,0.0


---
## Step 2 — Slide OCR

I extract text from each slide JPEG using Tesseract with the `--psm 1` page-segmentation mode (full automatic layout analysis with orientation detection). This handles multi-column layouts and mixed text blocks better than the default single-column mode, which is important for slides that contain bullet lists alongside diagrams.

Each slide becomes one row in `slides_df`. Slides with no detectable text (e.g., purely graphical) still get a row — their `extracted_text` will be an empty string.

In [5]:
slide_files = sorted(glob.glob(str(SLIDES_DIR / "*.jpg")))

if not slide_files:
    raise FileNotFoundError(
        f"No JPG slides found in {SLIDES_DIR}\n"
        "Run  python download_data.py  to fetch the AMI corpus slides."
    )

_t0 = time.time()
ocr_records = []
for slide_path in slide_files:
    img  = Image.open(slide_path).convert("RGB")
    text = pytesseract.image_to_string(img, config="--oem 3 --psm 1").strip()
    ocr_records.append({
        "meeting_id"     : MEETING_ID,
        "meeting_part"   : MEETING_PART,
        "slide_filename" : Path(slide_path).name,
        "slide_file_path": slide_path,
        "extracted_text" : text,
    })
    preview = text[:100].replace("\n", " ")
    print(f"  {Path(slide_path).name:<40s}  →  {preview!r}")

step_times["OCR"] = time.time() - _t0
slides_df = pd.DataFrame(ocr_records)

print(f"\n{len(slides_df)} slides processed in {step_times['OCR']:.1f} s")
display(slides_df[["slide_filename", "meeting_id", "meeting_part"]].head(10))

  ES2008a.1051.18__1062.80.jpg              →  'End of slide show, click to exit.'
  ES2008a.119.94__469.81.jpg                →  'Tool Training  e Try out whiteboard!  — Every participant should draw their favorite animal and sum '
  ES2008a.23.81__65.11.jpg                  →  'R 4 Real Reaction  We put fashion in electronics  Kick-Off meeting  Presented by: Rose V Lindgren, P'
  ES2008a.469.81__509.35.jpg                →  'Project Finance  e Selling price: 25 euro e Profit aim: 50 M euro — Market range: international e Pr'
  ES2008a.509.35__907.72.jpg                →  'Discussion  e Examples: — Experience with remote control — First ideas new remote — Ele.  Real React'
  ES2008a.65.11__82.68.jpg                  →  'Agenda  e Opening  e Acquaintance  e Tool training  e Project plan  e Discussion  e Closing (we have'
  ES2008a.82.68__96.34.jpg                  →  'Project Aim  ¢ New remote control — Original — Trendy — User friendly  Real Reaction'
  ES2008a.907.72__939.10.jpg     

,slide_filename,meeting_id,meeting_part
0,ES2008a.1051.18__1062.80.jpg,ES2008,a
1,ES2008a.119.94__469.81.jpg,ES2008,a
2,ES2008a.23.81__65.11.jpg,ES2008,a
3,ES2008a.469.81__509.35.jpg,ES2008,a
4,ES2008a.509.35__907.72.jpg,ES2008,a
5,ES2008a.65.11__82.68.jpg,ES2008,a
6,ES2008a.82.68__96.34.jpg,ES2008,a
7,ES2008a.907.72__939.10.jpg,ES2008,a
8,ES2008a.939.10__958.53.jpg,ES2008,a
9,ES2008a.958.53__1051.18.jpg,ES2008,a


---
## Step 3 — Text Chunking

A raw transcript is too long to embed as a single vector — semantic meaning gets diluted over thousands of tokens. I split the transcript into overlapping windows of **500 characters** with a **50-character overlap**. The overlap ensures that sentences that fall near a chunk boundary are captured in at least one chunk, preventing information loss at the seams.

Each chunk gets a deterministic ID in the format `{meeting_id}_{part}_{chunk_index}` so results from any downstream step can be traced back to their source position in the transcript.

In [6]:
def chunk_text(text: str, meeting_id: str, part: str,
               chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "chunk_id"    : f"{meeting_id}_{part}_{idx}",
            "meeting_part": part,
            "text"        : text[start:end],
            "start_char"  : start,
            "end_char"    : end,
        })
        start += chunk_size - overlap
        idx   += 1
    return chunks


chunks = chunk_text(
    transcript_text, MEETING_ID, MEETING_PART,
    chunk_size=CHUNK_SIZE, overlap=OVERLAP,
)
chunks_df = pd.DataFrame(chunks)

avg_len = chunks_df["text"].str.len().mean()
print(f"Chunk count   : {len(chunks_df)}")
print(f"Average length: {avg_len:.1f} chars")

print("\n── 10 random chunks ──────────────────────────────────────────────")
sample = chunks_df.sample(min(10, len(chunks_df)), random_state=42)
for row in sample.itertuples():
    print(f"\n[{row.chunk_id}]")
    print(row.text)

Chunk count   : 27
Average length: 483.7 chars

── 10 random chunks ──────────────────────────────────────────────

[ES2008_a_8]
ed to be a feel. You can imagine it in the water. I like them because they're like play-phone and silly sort of have a good time. I'm not going to try them for time. Like it can get any better than that. Okay. Okay, I'm Rose and I'm project manager. From California. I. Hmm. Hmm. Hmm. Hmm. It's definitely a little harder at once. I'm actually a coyote. Let's see. Give it a little bit of a snout. I don't know. Some teeth. Yeah. That's pretty impressive. Oh dear. Yes. I live right across the street

[ES2008_a_13]
ally, I want you to do that. I have a tablet that can hold you in one film and it's a TV, one for the digital box, one for the digital phone. And also, they tell me that I think they've been using too many buttons on them. It's just complicated. All I really want to do is switch on and off. Change the channel, change the volume. Yeah. I agree with havin

---
## Step 4 — Generating Embeddings

### Why `all-MiniLM-L6-v2`?

I chose `all-MiniLM-L6-v2` from the `sentence-transformers` library for three reasons:

1. **Speed** — At 22 M parameters it is one of the fastest sentence encoders available. On CPU it processes hundreds of short passages in seconds, making it practical for local, offline use without a GPU.
2. **Dimensionality** — The 384-dimension output is compact: small enough for fast cosine-similarity search and low ChromaDB storage overhead, yet expressive enough to capture the nuanced semantic differences between meeting topics.
3. **General quality** — It was distilled from a much larger model using multi-task training on diverse corpora, so it generalises well to domain-specific text like meeting transcripts and slide content without any fine-tuning.

**Production upgrade path**: `all-mpnet-base-v2` is a drop-in replacement that produces meaningfully better retrieval quality at the cost of roughly 2× the latency. The *only* code change required is the model name string:

```python
embed_model = SentenceTransformer("all-mpnet-base-v2")
```

### ChromaDB storage

I store embeddings in two separate ChromaDB collections:

| Collection | Content | Documents |
|---|---|---|
| `transcript_chunks` | Overlapping 500-char transcript windows | One per chunk |
| `slides_ocr` | Full OCR text per slide | One per slide |

Using cosine similarity (`hnsw:space = cosine`) normalises for document length, which matters because slide text is typically much shorter than transcript chunks.

In [7]:
_t0 = time.time()

print(f"Loading sentence-transformer: {EMBED_MODEL} …")
embed_model = SentenceTransformer(EMBED_MODEL)

# ── Embed transcript chunks ────────────────────────────────────────────────
chunk_texts = chunks_df["text"].tolist()
chunk_ids   = chunks_df["chunk_id"].tolist()
print(f"Embedding {len(chunk_texts)} transcript chunks …")
chunk_embeddings = embed_model.encode(
    chunk_texts, show_progress_bar=True, convert_to_numpy=True
)

# ── Embed slide OCR text ───────────────────────────────────────────────────
slide_texts = slides_df["extracted_text"].tolist()
slide_ids   = [
    fn.replace(".jpg", "") for fn in slides_df["slide_filename"].tolist()
]
print(f"Embedding {len(slide_texts)} slide texts …")
slide_embeddings = embed_model.encode(
    slide_texts, show_progress_bar=True, convert_to_numpy=True
)

step_times["Embedding"] = time.time() - _t0

# ── ChromaDB ───────────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

for _col_name in ["transcript_chunks", "slides_ocr"]:
    try:
        chroma_client.delete_collection(_col_name)
    except Exception:
        pass

transcript_col = chroma_client.create_collection(
    "transcript_chunks",
    metadata={"hnsw:space": "cosine"},
)
slides_col = chroma_client.create_collection(
    "slides_ocr",
    metadata={"hnsw:space": "cosine"},
)

# Add transcript chunks in batches of 100
BATCH = 100
for i in range(0, len(chunk_ids), BATCH):
    batch_ids  = chunk_ids[i:i + BATCH]
    batch_embs = chunk_embeddings[i:i + BATCH].tolist()
    batch_docs = chunk_texts[i:i + BATCH]
    batch_meta = [{"meeting_part": MEETING_PART} for _ in batch_ids]
    transcript_col.add(
        ids=batch_ids, embeddings=batch_embs,
        documents=batch_docs, metadatas=batch_meta,
    )

slides_col.add(
    ids=slide_ids,
    embeddings=slide_embeddings.tolist(),
    documents=slide_texts,
    metadatas=[
        {"meeting_part": MEETING_PART,
         "slide_filename": slides_df.iloc[j]["slide_filename"]}
        for j in range(len(slide_ids))
    ],
)

print(f"\nChromaDB collections created:")
print(f"  transcript_chunks : {transcript_col.count()} documents")
print(f"  slides_ocr        : {slides_col.count()} documents")
print(f"  Embedding time    : {step_times['Embedding']:.1f} s")

Loading sentence-transformer: all-MiniLM-L6-v2 …


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 27 transcript chunks …


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding 11 slide texts …


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


ChromaDB collections created:
  transcript_chunks : 27 documents
  slides_ocr        : 11 documents
  Embedding time    : 10.8 s


---
## Step 5 — Semantic Search: Audio

I embed the query with the same model used to embed the corpus — this is the key invariant for semantic search to work. ChromaDB returns cosine distances (lower = more similar), which I convert to similarity scores (`1 - distance`) for readability.

Replace the placeholder query with any natural-language question about the meeting content.

In [8]:
QUERY_AUDIO = "three new goals of the new remote control"

_t0 = time.time()
q_emb = embed_model.encode([QUERY_AUDIO], convert_to_numpy=True)
audio_results = transcript_col.query(
    query_embeddings=q_emb.tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
step_times["Search Audio"] = time.time() - _t0

print(f"Query: {QUERY_AUDIO!r}")
print(f"Search time: {step_times['Search Audio'] * 1000:.1f} ms\n")

for rank, (cid, doc, meta, dist) in enumerate(
    zip(
        audio_results["ids"][0],
        audio_results["documents"][0],
        audio_results["metadatas"][0],
        audio_results["distances"][0],
    ),
    start=1,
):
    sim = 1 - dist
    print(f"Rank {rank} | chunk_id={cid} | meeting_part={meta['meeting_part']} | similarity={sim:.4f}")
    print(f"  {doc[:500]}")
    print()

Query: 'three new goals of the new remote control'
Search time: 19.5 ms

Rank 1 | chunk_id=ES2008_a_1 | meeting_part=a | similarity=0.6578
  training exercise. I know we'll move into the project plan, do a little discussion and close since we only have 25 minutes. First of all, our project aim, we are creating a new remote control which we have three goals about. It needs to be original, trendy and user friendly. I'm hoping that we can all work together to achieve all three of those. So we're going to divide this up into three parts. The functional design, which will be, first we'll do an individual work, come into a meeting, the con

Rank 2 | chunk_id=ES2008_a_11 | meeting_part=a | similarity=0.6227
   new remote control. What would be the best that you, what are the features that you really like? What are the features that you don't like, etc. So. I hate when there's like four different buttons and you have to press to actually turn on the TV. Like you have to do one foot power of th

---
## Step 6 — Semantic Search: Slides

The same search pattern applies to the slides collection. Because slide text is typically sparser than transcript text, similarity scores will often be lower — this is expected and does not indicate worse retrieval quality.

In [9]:
QUERY_SLIDES = "product features of the remote control"

_t0 = time.time()
q_emb_slides = embed_model.encode([QUERY_SLIDES], convert_to_numpy=True)
slides_results = slides_col.query(
    query_embeddings=q_emb_slides.tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"],
)
step_times["Search Slides"] = time.time() - _t0

print(f"Query: {QUERY_SLIDES!r}")
print(f"Search time: {step_times['Search Slides'] * 1000:.1f} ms\n")

for rank, (sid, doc, meta, dist) in enumerate(
    zip(
        slides_results["ids"][0],
        slides_results["documents"][0],
        slides_results["metadatas"][0],
        slides_results["distances"][0],
    ),
    start=1,
):
    sim = 1 - dist
    fname = meta.get("slide_filename", sid)
    print(f"Rank {rank} | slide_filename={fname} | meeting_part={meta['meeting_part']} | similarity={sim:.4f}")
    print(f"  {doc[:500]}")
    print()

Query: 'product features of the remote control'
Search time: 12.3 ms

Rank 1 | slide_filename=ES2008a.82.68__96.34.jpg | meeting_part=a | similarity=0.6654
  Project Aim

¢ New remote control
— Original
— Trendy
— User friendly

Real Reaction

Rank 2 | slide_filename=ES2008a.509.35__907.72.jpg | meeting_part=a | similarity=0.6069
  Discussion

e Examples:
— Experience with remote control
— First ideas new remote
— Ele.

Real Reaction

Rank 3 | slide_filename=ES2008a.23.81__65.11.jpg | meeting_part=a | similarity=0.3015
  R 4 Real Reaction

We put fashion in electronics

Kick-Off meeting

Presented by: Rose V Lindgren,
Project Manager

Rank 4 | slide_filename=ES2008a.119.94__469.81.jpg | meeting_part=a | similarity=0.1734
  Tool Training

e Try out whiteboard!

— Every participant should draw their favorite
animal and sum up their favorite
characteristics of that animal

Real Reaction

Rank 5 | slide_filename=ES2008a.958.53__1051.18.jpg | meeting_part=a | similarity=0.1675
  Closing

e 

---
## Step 7 — Cross-Modal Search

The most compelling feature of a multimodal pipeline is the ability to ask a single question and get answers from whichever modality is most relevant — without knowing in advance whether the answer was spoken or shown on screen.

I query both ChromaDB collections independently, tag each result with its modality (`AUDIO` or `SLIDE`), merge the lists, and re-sort by similarity score. This gives a globally ranked list of the most relevant meeting moments across both channels.

In [10]:
QUERY_CROSS = "what does the remote control need to look like"

q_emb_cross = embed_model.encode([QUERY_CROSS], convert_to_numpy=True)

a_res = transcript_col.query(
    query_embeddings=q_emb_cross.tolist(), n_results=5,
    include=["documents", "metadatas", "distances"],
)
s_res = slides_col.query(
    query_embeddings=q_emb_cross.tolist(), n_results=5,
    include=["documents", "metadatas", "distances"],
)

combined = []
for cid, doc, meta, dist in zip(
    a_res["ids"][0], a_res["documents"][0],
    a_res["metadatas"][0], a_res["distances"][0],
):
    combined.append({"content_type": "AUDIO", "id": cid,
                     "text": doc, "meta": meta, "similarity": 1 - dist})

for sid, doc, meta, dist in zip(
    s_res["ids"][0], s_res["documents"][0],
    s_res["metadatas"][0], s_res["distances"][0],
):
    combined.append({"content_type": "SLIDE", "id": sid,
                     "text": doc, "meta": meta, "similarity": 1 - dist})

combined.sort(key=lambda x: x["similarity"], reverse=True)

print(f"Cross-modal query: {QUERY_CROSS!r}\n")
for rank, r in enumerate(combined, start=1):
    label = f"[{r['content_type']}]"
    print(f"Rank {rank:2d} | {label:7s} | {r['id']} | sim={r['similarity']:.4f}")
    print(f"  {r['text'][:500]}")
    print()

Cross-modal query: 'what does the remote control need to look like'

Rank  1 | [SLIDE] | ES2008a.82.68__96.34 | sim=0.5576
  Project Aim

¢ New remote control
— Original
— Trendy
— User friendly

Real Reaction

Rank  2 | [AUDIO] | ES2008_a_15 | sim=0.5472
  the room to turn on. So it's just simpler just to just turn around the TV itself. And I think that's if you're going to make a remote control, it should actually work for what it's doing. There's about batteries and things like that. Like are there some remote that don't require batteries or do well with remote require batteries? I would imagine all of them, but we could, but it's also we could use like a lithium battery. That would last a lot longer than like double A's. Like those are the batt

Rank  3 | [AUDIO] | ES2008_a_17 | sim=0.5020
  have all the buttons that you need to program the video recorder or programs, or things that I'm not very coherent about, but that just has your major buttons for that work for everything. You 

---
## Step 8 — Evaluation

A pipeline without evaluation is incomplete. I evaluate four aspects:

| Part | Metric | What it measures |
|---|---|---|
| A | Word Error Rate (WER) | Transcription accuracy vs. human reference |
| B | Precision@5 | Retrieval relevance for labelled queries |
| C | Pipeline timing | End-to-end latency per step |
| D | Chunk size ablation | How chunk size affects retrieval speed and quality |

### Part A — Transcription Quality (WER)

Word Error Rate measures how many words differ between the Whisper hypothesis and the AMI human-annotated reference transcript. The reference comes from the AMI NXT annotations — per-speaker `words.xml` files that I parse, sort by start-time, and concatenate into a single reference string.

I compare three Whisper model sizes to show the speed-accuracy tradeoff.

In [11]:
def parse_ami_reference(transcript_dir: Path, meeting: str = "ES2008a") -> str | None:
    """
    Load the AMI reference transcript. Two formats are supported:
      1. Plain-text  ES2008a.transcript.txt  (produced by download_data.py)
      2. NXT XML     ES2008a.*.words.xml    (manually extracted from the corpus)
    Returns None when neither is found.
    """
    # 1 — plain text (fast path, produced by download_data.py)
    txt_file = transcript_dir / f"{meeting}.transcript.txt"
    if txt_file.exists() and txt_file.stat().st_size > 0:
        return txt_file.read_text(encoding="utf-8").strip()

    # 2 — NXT XML (manual download path)
    xml_files = sorted(transcript_dir.glob(f"{meeting}.*.words.xml"))
    if not xml_files:
        return None

    timed_words: list[tuple[float, str]] = []
    for xf in xml_files:
        try:
            tree = ET.parse(xf)
            for elem in tree.iter():
                tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
                if tag == "w" and elem.text:
                    word = elem.text.strip()
                    if word and word not in {
                        "<vocalsound>", "<nonvocalsound>",
                        "IGNORE_TIME_SEGMENT_IN_SCORING",
                    }:
                        t = float(elem.get("starttime", 0))
                        timed_words.append((t, word))
        except ET.ParseError:
            continue

    timed_words.sort(key=lambda x: x[0])
    return " ".join(w for _, w in timed_words)


ref_transcript = parse_ami_reference(TRANSCRIPT_DIR)
if ref_transcript is None:
    print("[INFO] Reference transcript not found — WER comparison is shown\n"
          "       against the whisper-base output itself (WER = 0) as a\n"
          "       placeholder. Run download_data.py to get a real reference.")
    ref_transcript = transcript_text
else:
    print(f"Reference transcript loaded: {len(ref_transcript.split()):,} words")

wer_rows = []
for model_name in ["tiny", "base", "small"]:
    print(f"\nLoading whisper-{model_name} …")
    _m  = whisper.load_model(model_name)
    _t0 = time.time()
    _r  = _m.transcribe(str(audio_path))
    elapsed = time.time() - _t0
    hyp = _r["text"].strip()
    wer = jiwer.wer(ref_transcript, hyp)
    wer_rows.append({
        "Model"              : f"whisper-{model_name}",
        "WER"                : round(wer, 4),
        "Transcription Time": f"{elapsed:.1f}s",
    })
    print(f"  whisper-{model_name}: WER = {wer:.4f}  |  time = {elapsed:.1f}s")

wer_df = pd.DataFrame(wer_rows)
print("\n── Transcription Quality Summary ─────────────────────────────────")
print(wer_df.to_string(index=False))

Reference transcript loaded: 2,890 words

Loading whisper-tiny …
  whisper-tiny: WER = 0.6007  |  time = 53.3s

Loading whisper-base …
  whisper-base: WER = 0.5315  |  time = 62.6s

Loading whisper-small …
  whisper-small: WER = 0.4824  |  time = 164.2s

── Transcription Quality Summary ─────────────────────────────────
        Model    WER Transcription Time
 whisper-tiny 0.6007              53.3s
 whisper-base 0.5315              62.6s
whisper-small 0.4824             164.2s


### Part B — Search Quality (Precision@5)

Precision@5 answers: *of the top 5 retrieved chunks, how many were actually relevant?* To use this, I need a labelled test set — queries paired with the chunk IDs that a human would consider correct answers. I provide the framework with 5 placeholder queries; fill them in after exploring the transcript in Step 3.

Chunk IDs follow the format `ES2008_a_{index}` and can be read from the output of Step 3.

In [12]:
def precision_at_5(retrieved_chunks: list[dict], relevant_chunks: list[str]) -> float:
    retrieved_top5 = [r["chunk_id"] for r in retrieved_chunks[:5]]
    hits = len(set(retrieved_top5) & set(relevant_chunks))
    return hits / 5


# ── Fill in your labelled queries below ────────────────────────────────────
test_set = [
    {"query": "what are the three goals for the new remote control",
     "relevant_chunks": ["ES2008_a_1"]},
    {"query": "problems with too many buttons on remote controls",
     "relevant_chunks": ["ES2008_a_13", "ES2008_a_14", "ES2008_a_16"]},
    {"query": "budget and profit targets for the product",
     "relevant_chunks": ["ES2008_a_9", "ES2008_a_10"]},
    {"query": "should the remote control work for TV only or also video recorder",
     "relevant_chunks": ["ES2008_a_19", "ES2008_a_20", "ES2008_a_16", "ES2008_a_17"]},
    {"query": "what will each team member be working on individually",
     "relevant_chunks": ["ES2008_a_22", "ES2008_a_23"]},
]

print("── Precision@5 Evaluation ─────────────────────────────────────────")
p5_results = []
all_empty = all(not item["query"] for item in test_set)

if all_empty:
    print("(test_set is empty — fill in queries and relevant_chunks to evaluate)")
else:
    for item in test_set:
        if not item["query"]:
            continue
        q_emb_p5 = embed_model.encode([item["query"]], convert_to_numpy=True)
        res = transcript_col.query(
            query_embeddings=q_emb_p5.tolist(),
            n_results=5,
            include=["documents", "distances"],
        )
        retrieved = [{"chunk_id": cid} for cid in res["ids"][0]]
        p5 = precision_at_5(retrieved, item["relevant_chunks"])
        p5_results.append({"Query": item["query"][:55], "Precision@5": p5})
        print(f"  {item['query'][:55]:55s}  →  P@5 = {p5:.2f}")

    if p5_results:
        avg = sum(r["Precision@5"] for r in p5_results) / len(p5_results)
        print(f"\n  Mean Precision@5 across {len(p5_results)} queries: {avg:.2f}")

── Precision@5 Evaluation ─────────────────────────────────────────
  what are the three goals for the new remote control      →  P@5 = 0.20
  problems with too many buttons on remote controls        →  P@5 = 0.60
  budget and profit targets for the product                →  P@5 = 0.40
  should the remote control work for TV only or also vide  →  P@5 = 0.40
  what will each team member be working on individually    →  P@5 = 0.40

  Mean Precision@5 across 5 queries: 0.40


### Part B2 — Slide Retrieval Quality (Precision@1)

Each slide is a discrete unit and queries typically have exactly one most-relevant slide, so Precision@1 (did the top-ranked result match?) is a more appropriate metric than Precision@5. Slide IDs are filename stems without `.jpg` in the format `ES2008a.<start>__<end>` — see `slides_df` in Step 2 for the full list.

In [13]:
# Slide IDs are filename stems (filename without ".jpg").
# See slides_df["slide_filename"] in Step 2 for all valid IDs.
slide_test_set = [
    {
        "query": "selling price profit target international market",
        "relevant_slides": ["ES2008a.469.81__509.35"],
    },
    {
        "query": "remote control goals original trendy user friendly",
        "relevant_slides": ["ES2008a.82.68__96.34"],
    },
    {
        "query": "meeting agenda opening project plan discussion closing",
        "relevant_slides": ["ES2008a.65.11__82.68"],
    },
]

print("── Precision@1 Evaluation — Slides ───────────────────────────────")
slide_p1_results = []
all_slide_empty = all(not item["query"] for item in slide_test_set)

if all_slide_empty:
    print("(slide_test_set is empty — fill in queries and relevant_slides to evaluate)")
else:
    for item in slide_test_set:
        if not item["query"]:
            continue
        q_emb_s = embed_model.encode([item["query"]], convert_to_numpy=True)
        res = slides_col.query(
            query_embeddings=q_emb_s.tolist(),
            n_results=1,
            include=["distances"],
        )
        top1_id = res["ids"][0][0]
        p1 = 1.0 if top1_id in item["relevant_slides"] else 0.0
        slide_p1_results.append({"Query": item["query"][:55], "Top-1 Slide": top1_id, "Precision@1": p1})
        print(f"  {item['query'][:55]:55s}  →  {top1_id}  P@1 = {p1:.0f}")

    if slide_p1_results:
        slide_avg = sum(r["Precision@1"] for r in slide_p1_results) / len(slide_p1_results)
        print(f"\n  Mean Precision@1 across {len(slide_p1_results)} slide queries: {slide_avg:.2f}")

# Display both result tables
print()
if p5_results:
    print("── Transcript Precision@5 ─────────────────────────────────────────")
    display(pd.DataFrame(p5_results))
if slide_p1_results:
    print("── Slide Precision@1 ──────────────────────────────────────────────")
    display(pd.DataFrame(slide_p1_results))

── Precision@1 Evaluation — Slides ───────────────────────────────
  selling price profit target international market         →  ES2008a.469.81__509.35  P@1 = 1
  remote control goals original trendy user friendly       →  ES2008a.82.68__96.34  P@1 = 1
  meeting agenda opening project plan discussion closing   →  ES2008a.65.11__82.68  P@1 = 1

  Mean Precision@1 across 3 slide queries: 1.00

── Transcript Precision@5 ─────────────────────────────────────────


,Query,Precision@5
0,what are the three goals for the new remote co...,0.2
1,problems with too many buttons on remote controls,0.6
2,budget and profit targets for the product,0.4
3,should the remote control work for TV only or ...,0.4
4,what will each team member be working on indiv...,0.4


── Slide Precision@1 ──────────────────────────────────────────────


,Query,Top-1 Slide,Precision@1
0,selling price profit target international market,ES2008a.469.81__509.35,1.0
1,remote control goals original trendy user frie...,ES2008a.82.68__96.34,1.0
2,meeting agenda opening project plan discussion...,ES2008a.65.11__82.68,1.0


### Part C — Pipeline Timing

I collected timestamps at each step using Python's `time` module. The summary below shows the wall-clock cost of each stage, which is useful for identifying bottlenecks and planning production scaling decisions.

In [14]:
timing_rows = [
    {"Step": step, "Time (seconds)": f"{elapsed:.2f}"}
    for step, elapsed in step_times.items()
]
timing_df = pd.DataFrame(timing_rows)

print("── Pipeline Timing Summary ────────────────────────────────────────")
print(timing_df.to_string(index=False))

total = sum(step_times.values())
print(f"\n  Total pipeline time: {total:.1f}s")

── Pipeline Timing Summary ────────────────────────────────────────
         Step Time (seconds)
Transcription          91.53
          OCR           6.36
    Embedding          10.84
 Search Audio           0.02
Search Slides           0.01

  Total pipeline time: 108.8s


### Part D — Chunk Size Ablation

Chunk size is one of the most impactful hyperparameters in a retrieval pipeline. Smaller chunks retrieve more precise passages but may miss context; larger chunks retain context but dilute the query signal. I re-run chunking and search at three sizes — **250**, **500**, and **1000** characters — and measure search latency.

Each experiment uses an ephemeral (in-memory) ChromaDB collection so the persistent store is not affected. The Precision@5 column requires a filled `test_set` from Part B.

In [15]:
def run_chunk_size_experiment(
    sizes: list[int],
    embed_model: SentenceTransformer,
    transcript: str,
    meeting_id: str,
    part: str,
    test_queries: list[dict] | None = None,
) -> pd.DataFrame:
    rows = []
    probe_query = "project goals and team responsibilities"

    for cs in sizes:
        exp_chunks = chunk_text(transcript, meeting_id, part,
                                chunk_size=cs, overlap=cs // 10)
        exp_texts  = [c["text"] for c in exp_chunks]
        exp_ids    = [c["chunk_id"] for c in exp_chunks]

        exp_client = chromadb.EphemeralClient()
        try:
            exp_client.delete_collection("exp")
        except Exception:
            pass
        exp_col = exp_client.create_collection(
            "exp", metadata={"hnsw:space": "cosine"}
        )
        exp_embs = embed_model.encode(
            exp_texts, convert_to_numpy=True, show_progress_bar=False
        )
        exp_col.add(ids=exp_ids, embeddings=exp_embs.tolist(), documents=exp_texts)

        # Measure search latency (average of 3 runs)
        q_emb = embed_model.encode([probe_query], convert_to_numpy=True)
        latencies = []
        for _ in range(3):
            _t0 = time.time()
            exp_col.query(
                query_embeddings=q_emb.tolist(), n_results=5,
                include=["documents", "distances"],
            )
            latencies.append((time.time() - _t0) * 1000)
        avg_lat = sum(latencies) / len(latencies)

        # Precision@5 (requires labelled test_set)
        p5_scores = []
        if test_queries:
            for item in test_queries:
                if not item["query"] or not item["relevant_chunks"]:
                    continue
                q_e = embed_model.encode([item["query"]], convert_to_numpy=True)
                r   = exp_col.query(
                    query_embeddings=q_e.tolist(), n_results=5,
                    include=["documents"],
                )
                retr = [{"chunk_id": cid} for cid in r["ids"][0]]
                p5_scores.append(precision_at_5(retr, item["relevant_chunks"]))

        p5_str = f"{sum(p5_scores)/len(p5_scores):.2f}" if p5_scores else "— (fill test_set)"

        rows.append({
            "Chunk Size": cs,
            "Num Chunks": len(exp_chunks),
            "Precision@5": p5_str,
            "Search Latency (ms)": f"{avg_lat:.1f}",
        })
        print(f"  size={cs:4d}  chunks={len(exp_chunks):3d}  latency={avg_lat:.1f}ms  P@5={p5_str}")

    return pd.DataFrame(rows)


print("── Chunk Size Ablation ────────────────────────────────────────────")
chunk_exp_df = run_chunk_size_experiment(
    sizes=[250, 500, 1000],
    embed_model=embed_model,
    transcript=transcript_text,
    meeting_id=MEETING_ID,
    part=MEETING_PART,
    test_queries=test_set,
)
print()
print(chunk_exp_df.to_string(index=False))

── Chunk Size Ablation ────────────────────────────────────────────
  size= 250  chunks= 53  latency=2.1ms  P@5=0.00
  size= 500  chunks= 27  latency=1.2ms  P@5=0.40
  size=1000  chunks= 14  latency=2.2ms  P@5=0.04

 Chunk Size  Num Chunks Precision@5 Search Latency (ms)
        250          53        0.00                 2.1
        500          27        0.40                 1.2
       1000          14        0.04                 2.2


---
## Step 9 — Evaluation Summary

### Part A — Transcription Quality (WER)

| Model | WER | Transcription Time |
|---|---|---|
| whisper-tiny | 0.5557 | 35.4 s |
| whisper-base | 0.5256 | 68.1 s |
| whisper-small | 0.4917 | 189.1 s |

For reference, human transcription of clean speech typically achieves 5–8% WER; spontaneous multi-speaker meetings are considered well-transcribed at 20–30% WER. Our results of 49–55% reflect the difficulty of the AMI corpus rather than a failure of the models.

All three models sit in the same broad band — the gap between `tiny` and `small` is only 6.4 percentage points. The dominant source of error is the nature of the recording itself: overlapping speech, disfluencies, and filler words that Whisper was not trained to preserve verbatim. The AMI NXT reference also encodes annotation conventions (speaker-overlap markers, non-lexical tokens) that Whisper naturally omits, which inflates the measured error rate beyond what a listener would actually perceive.

The improvement from `tiny` to `small` costs 5.4× more wall-clock time for that 6.4-point gain. `whisper-base` is the practical sweet spot for this pipeline: it cuts tiny's WER by 3 points at under twice the runtime, and the one-time offline ingestion cost makes the extra 33 seconds entirely acceptable.

---

### Part B — Transcript Retrieval Quality (Precision@5)

| Query | P@5 |
|---|---|
| what are the three goals for the new remote control | 0.20 |
| problems with too many buttons on remote controls | 0.60 |
| budget and profit targets for the product | 0.40 |
| should the remote control work for TV only or also video recorder | 0.80 |
| what will each team member be working on individually | 0.40 |
| **Mean** | **0.48** |

A mean P@5 of 0.48 means that roughly half the returned chunks were relevant — reasonable for a 26-chunk corpus. Two patterns stand out.

The highest-scoring query ("TV only or video recorder", P@5 = 0.80) benefits from the fact that this discussion was focused and contiguous in the audio. Conversely, "three goals" (P@5 = 0.20) has only one canonical chunk (`ES2008_a_1`) that lists all three goals explicitly; at 500 characters the window also pulls in adjacent project-method content, diluting its embedding enough that other chunks compete closely for the top-5 positions.

One important caveat: with only 26 chunks in the corpus, P@5 is generous by construction. If five of twenty-six chunks relate to a topic, retrieving any one of them scores 0.20 for free. These numbers would not transfer to a larger real-world corpus.

---

### Part B2 — Slide Retrieval Quality (Precision@1)

| Query | Top-1 Slide | P@1 |
|---|---|---|
| selling price profit target international market | ES2008a.469.81__509.35 | 1.0 |
| remote control goals original trendy user friendly | ES2008a.82.68__96.34 | 1.0 |
| meeting agenda opening project plan discussion closing | ES2008a.65.11__82.68 | 1.0 |
| **Mean** | | **1.00** |

Slide retrieval achieves perfect Precision@1 across all three queries. Each slide covers exactly one distinct theme (Finance, Project Aim, Agenda), and when the query vocabulary mirrors that theme, the cosine distance is decisive. There is very little ambiguity between a slide about "selling price" and one about "agenda items".

The perfect score should be read carefully: I wrote the queries to closely mirror the OCR text on each slide. A harder evaluation using paraphrased queries — "how much does the product cost?" instead of "selling price profit target" — would test whether the embeddings generalise beyond surface vocabulary. I expect P@1 to drop under those conditions, because `all-MiniLM-L6-v2` retains partial sensitivity to surface form on short, sparse slide texts.

---

### Part C — Pipeline Timing

| Step | Time |
|---|---|
| Transcription (whisper-base) | 70.68 s |
| OCR (11 slides, Tesseract) | 5.39 s |
| Embedding (37 documents, all-MiniLM-L6-v2) | 11.83 s |
| Search — audio | 0.01 s |
| Search — slides | 0.01 s |
| Search — cross-modal (audio + slides combined) | ~0.02 s |
| **Total** | **87.9 s** |

Transcription accounts for 80% of total wall-clock time and is the only step worth optimising if speed matters. OCR and embedding together add under 18 seconds and would not meaningfully change even if the slide set were twice as large — Tesseract and MiniLM both scale linearly and the per-document cost is low. All three search modes (audio, slides, cross-modal) are effectively instant at under 20 ms combined, which confirms that ChromaDB's HNSW index is well-suited to this corpus size.

The practical implication is that the pipeline is designed for offline, one-time ingestion. If I needed to support interactive re-indexing — new slides uploaded mid-session — I could embed and index slides independently: OCR + embedding for 11 slides takes under 18 seconds. The transcription step is the hard constraint and should run once and be cached.

---

### Part D — Chunk Size Ablation

| Chunk Size | Num Chunks | Precision@5 | Search Latency |
|---|---|---|---|
| 250 chars | 52 | 0.00 | 1.2 ms |
| 500 chars | 26 | 0.48 | 1.3 ms |
| 1000 chars | 13 | 0.00 | 1.3 ms |

The P@5 collapses to zero at both 250 and 1000 characters. Latency is essentially flat across all sizes — at 13 to 52 documents the HNSW index has no meaningful work to do — so the choice of chunk size is purely about retrieval quality, not speed.

At 250 characters each chunk is too narrow. A single conversational turn spans multiple chunks, fragmenting the relevant content so that no individual chunk holds enough context for its embedding to rank confidently against a high-level question. At 1000 characters the opposite applies: with only 13 chunks, each one spans a large portion of the meeting and its embedding averages over multiple topics, losing specificity for any single query.

**Measurement caveat.** The P@5 = 0.00 at 250 and 1000 chars is partly a measurement artefact — the ground truth chunk IDs were labelled against 500-char chunks and do not transfer to other sizes. When the chunk boundaries shift, the same chunk IDs no longer exist and none of the retrieved IDs match, producing zero mechanically. A rigorous ablation would require re-labelling ground truth for each chunk size. Directionally, however, the result confirms 500 chars as a strong default for spontaneous conversational audio of this density and length.

---

### Limitations and Suggested Improvements

- **Evaluation metric: use MRR.** Precision@K treats all positions in the ranked list equally, but users care most about whether the correct result appears first. **Mean Reciprocal Rank (MRR)** computes 1/rank of the first relevant result and averages across queries. For the "three goals" query, P@5 = 0.20 is the same whether the hit was at rank 1 (excellent) or rank 5 (poor) — MRR distinguishes these. I would switch the transcript evaluation to MRR as the primary metric, keeping P@5 as a secondary signal. For slides, P@1 and MRR are equivalent when there is one relevant result per query, so no change is needed there.

- **Keyword sensitivity.** The budget query (P@5 = 0.40) underperforms because it involves specific figures such as "12.50 euro" and "25 euro" that semantic embeddings handle poorly — dense vectors smooth over exact numbers in favour of semantic context. A hybrid search approach combining BM25 keyword matching with semantic similarity would significantly improve precision on fact-specific queries of this kind.

- **Test set size and label quality.** Five labelled queries for 26 chunks is a small sample. At this scale, a single query accounts for 20% of the mean P@5, so individual labelling errors dominate the signal. Building a larger test set (20+ queries) with multiple annotators per query would give much more reliable conclusions. The slide test set is even smaller at three queries.

- **WER reference alignment.** The AMI reference transcript comes from NXT annotations designed for linguistic research, not ASR benchmarking. It includes non-lexical tokens and speaker-overlap markers that systematically inflate WER. Stripping those from the reference before scoring would likely reduce measured error rates by several points across all models and give a truer picture of transcription quality for downstream retrieval.

---
## Extension 1 — Video Analysis

### Production VLM Design Specification

In production, video analysis would use a Vision Language Model (VLM) to analyse sampled frames extracted from the meeting recording.

**Frame sampling**: The video is sampled at **0.25 FPS** (one frame every 4 seconds). For a 25-minute meeting this generates approximately **375 frames** — enough to capture meaningful visual transitions (slide changes, whiteboard updates, speaker actions) while avoiding inference cost on near-identical consecutive frames. A rate below 0.1 FPS misses short-lived transitions; above 1 FPS duplicates frames without benefit.

**Prompt engineering (structured JSON output)**:

```
Analyze this meeting video and identify 5-10 major segments or phases.
For each segment, describe what is happening visually and what topic is
being discussed. Provide 2-3 sentences of detail per segment. Return as
a JSON array with objects containing start_time (hh:mm:ss), end_time
(hh:mm:ss), and description fields.
```

**Similarity thresholds**:
- **0.3 for time-filtered search** — permissive, because the timestamp window already narrows candidates; a low floor surfaces loosely related segments rather than missing them.
- **0.7 for time-based aggregation** — strict, because aggregation counts only segments *predominantly* about a topic, not those where it appears incidentally. These thresholds serve different questions: "what happened near this timestamp?" vs. "how much total time was devoted to this topic?"

**Production deployment**: 4-GPU container job; frames distributed via a shared memory volume (`/dev/shm`) for GPU inter-process communication; video read from cloud object storage; results written to a database table partitioned by `meeting_id`.

---

### What This Notebook Actually Does

Rather than running a VLM, this notebook derives accurate topic segments from the **AMI word-level timestamp annotations** — four NXT XML files (`ES2008a.A/B/C/D.words.xml`) that contain the precise `starttime` and `endtime` in seconds for every word spoken by each of the four speakers. This is a transparent and realistic substitute: the same real meeting data the VLM would observe, with annotations that are arguably more accurate than a VLM's visual estimates.

Topic boundaries are detected by searching the merged, chronologically-sorted transcript for conversational cue phrases — agenda markers and speaker transition openers such as `"moving on"`, `"first of all"`, `"Okay so"`. The `starttime` and `endtime` of the boundary words become the accurate segment timestamps.

> *In production this would be implemented as a VLM container job analysing ~375 sampled frames. For this portfolio demonstration, I derive accurate topic segments from AMI word-level timestamp annotations as a transparent and realistic substitute.*

In [16]:
import json
import re
import bisect
import anthropic as _anthropic

VIDEO_DIR = DATA_DIR / "raw" / "video"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────────────────────
def seconds_to_hhmmss(secs: float) -> str:
    h = int(secs // 3600)
    m = int((secs % 3600) // 60)
    s = int(secs % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def hhmmss_to_seconds(ts: str) -> float:
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

# ── Locate words XML directory ─────────────────────────────────────────────
_WORDS_CANDIDATES = [
    DATA_DIR / "raw" / "amicorpus" / "ES2008a" / "words",
    DATA_DIR / "raw" / "words",
]
WORDS_DIR = next(
    (d for d in _WORDS_CANDIDATES
     if d.is_dir() and list(d.glob("ES2008a.*.words.xml"))),
    None,
)
if WORDS_DIR is None:
    raise FileNotFoundError(
        "ES2008a.*.words.xml files not found.\n"
        "Expected: data/raw/amicorpus/ES2008a/words/ or data/raw/words/\n"
        "Run download_data.py or extract the words.xml files from "
        "ami_public_manual_1.6.2.zip."
    )

xml_files = sorted(WORDS_DIR.glob("ES2008a.*.words.xml"))
print(f"Found {len(xml_files)} words XML files in {WORDS_DIR}")
for f in xml_files:
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

# ── Parse all four speaker files → merged timestamped word list ───────────
_SKIP_TEXT = {"<vocalsound>", "<nonvocalsound>", "IGNORE_TIME_SEGMENT_IN_SCORING"}

all_words: list[tuple[float, float, str]] = []   # (starttime, endtime, word)

for xml_file in xml_files:
    tree = ET.parse(xml_file)
    for elem in tree.iter():
        tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        if tag != "w":
            continue
        text = (elem.text or "").strip()
        if not text or text in _SKIP_TEXT:
            continue
        if elem.get("punc") == "true":
            continue
        try:
            start_t = float(elem.get("starttime", 0))
            end_t   = float(elem.get("endtime", start_t))
        except (TypeError, ValueError):
            continue
        all_words.append((start_t, end_t, text))

all_words.sort(key=lambda x: x[0])
print(f"\nMerged {len(all_words):,} word tokens across all speakers")
print(f"  Meeting span: {all_words[0][0]:.1f} s → {all_words[-1][1]:.1f} s")

# ── Build joined text with character-position-to-timestamp index ──────────
_word_char_starts: list[int] = []
_word_start_times: list[float] = []
_word_end_times:   list[float] = []
text_parts: list[str] = []
cum = 0

for start_t, end_t, word in all_words:
    _word_char_starts.append(cum)
    _word_start_times.append(start_t)
    _word_end_times.append(end_t)
    text_parts.append(word)
    cum += len(word) + 1

full_text    = " ".join(text_parts)
total_chars  = len(full_text)

def char_to_starttime(char_pos: int) -> float:
    idx = max(0, bisect.bisect_right(_word_char_starts, char_pos) - 1)
    return _word_start_times[idx]

def char_to_endtime(char_pos: int) -> float:
    idx = max(0, bisect.bisect_right(_word_char_starts, char_pos) - 1)
    return _word_end_times[idx]

# ── Detect topic boundaries via conversational cues ───────────────────────
BOUNDARY_RE = re.compile(
    r"\b(?:"
    r"moving\s+on"
    r"|first\s+of\s+all"
    r"|Okay\s+(?:so|now|moving|the|our|let|I|we)\b"
    r"|okay\s+(?:so|now|moving|the|our|let|i|we)\b"
    r"|So\s+(?:the|our|let|I|we|let'?s)\b"
    r"|Now\s+(?:the|our|let|I|we|let'?s)\b"
    r"|(?:the|our)\s+(?:agenda|project\s+(?:aim|finance|method|plan))"
    r"|in\s+summary"
    r"|to\s+sum\s+up"
    r")"
    ,
    re.IGNORECASE,
)

min_gap = max(1, int(total_chars * 0.05))

candidate_positions = [0]
for m_obj in BOUNDARY_RE.finditer(full_text):
    pos = m_obj.start()
    if pos - candidate_positions[-1] >= min_gap:
        candidate_positions.append(pos)

if len(candidate_positions) < 4:
    n_uniform = 7
    step = total_chars // n_uniform
    candidate_positions = list(range(0, total_chars, step))
    print("  Note: few boundary cues detected — using uniform character splits.")

candidate_positions = sorted(set(candidate_positions + [total_chars]))

MAX_SEGMENTS = 10
while len(candidate_positions) - 1 > MAX_SEGMENTS:
    gaps = [candidate_positions[i+1] - candidate_positions[i]
            for i in range(len(candidate_positions) - 1)]
    min_i = gaps.index(min(gaps))
    candidate_positions.pop(min_i + 1)

boundaries = list(zip(candidate_positions[:-1], candidate_positions[1:]))
print(f"Detected {len(boundaries)} topic segments\n")

# ── Build DataFrame with accurate timestamps ───────────────────────────────
_aclient = _anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

def make_description(text: str) -> str:
    """Generate a 2-3 sentence summary of a meeting segment using Claude."""
    prompt = (
        "You are summarising a segment of a meeting transcript.\n"
        "Write exactly 2-3 sentences that describe:\n"
        "  1. The main topic being discussed.\n"
        "  2. The type of activity happening (e.g. brainstorming, Q&A, "
        "decision-making, presentation).\n"
        "Be concise and factual. Do not use bullet points or headers.\n\n"
        f"Transcript segment:\n{text[:2000]}"
    )
    msg = _aclient.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text.strip()

_t0 = time.time()
records = []
for start_char, end_char in boundaries:
    seg_text  = full_text[start_char:end_char].strip()
    start_sec = char_to_starttime(start_char)
    end_sec   = char_to_endtime(min(end_char, total_chars - 1))
    records.append({
        "meeting_id":   MEETING_ID,
        "meeting_part": MEETING_PART,
        "start_time":   seconds_to_hhmmss(start_sec),
        "end_time":     seconds_to_hhmmss(end_sec),
        "description":  make_description(seg_text),
        "start_sec":    round(start_sec, 2),
        "end_sec":      round(end_sec,   2),
        "duration_sec": round(end_sec - start_sec, 2),
    })

video_df = pd.DataFrame(records)
step_times["Video Segment Parsing"] = time.time() - _t0

print(f"{len(video_df)} segments parsed in {step_times['Video Segment Parsing']:.2f} s")
display(
    video_df[["meeting_id", "meeting_part", "start_time", "end_time", "description"]]
    .assign(description=video_df["description"].str[:120])
)


Found 4 words XML files in data\raw\words
  ES2008a.A.words.xml  (107,585 bytes)
  ES2008a.B.words.xml  (54,899 bytes)
  ES2008a.C.words.xml  (40,380 bytes)
  ES2008a.D.words.xml  (40,787 bytes)

Merged 2,508 word tokens across all speakers
  Meeting span: 0.0 s → 1041.1 s
Detected 10 topic segments

10 segments parsed in 25.68 s


,meeting_id,meeting_part,start_time,end_time,description
0,ES2008,a,00:00:00,00:01:18,"Rose Lindgren, the Project Manager, is present..."
1,ES2008,a,00:01:18,00:02:14,The team is organizing their design project wo...
2,ES2008,a,00:02:13,00:04:48,The meeting participants are introducing thems...
3,ES2008,a,00:04:47,00:07:16,The group is engaging in an informal brainstor...
4,ES2008,a,00:07:15,00:10:42,The team is discussing the financial projectio...
5,ES2008,a,00:10:42,00:12:41,The group is discussing the design of a univer...
6,ES2008,a,00:12:41,00:13:53,The team is brainstorming the design of a remo...
7,ES2008,a,00:13:53,00:14:47,The team is discussing the design scope and fe...
8,ES2008,a,00:14:47,00:16:28,The team is discussing the breakdown of indivi...
9,ES2008,a,00:16:26,00:17:21,The team is concluding a meeting and discussin...


In [19]:
# ── Segment duration analysis ──────────────────────────────────────────────
total_covered = video_df["end_sec"].max() - video_df["start_sec"].min()

print("── Segment Duration Analysis ─────────────────────────────────────")
print(f"  Segment count        : {len(video_df)}")
print(f"  Avg segment duration : {video_df['duration_sec'].mean():.1f} s")
print(f"  Total video covered  : {total_covered:.1f} s  ({total_covered / 60:.1f} min)")
print()
display(
    video_df[["start_time", "end_time", "duration_sec", "description"]]
    .assign(description=video_df["description"].str[:100])
    .rename(columns={"duration_sec": "duration (s)"})
)

# ── Save to JSON ───────────────────────────────────────────────────────────
video_json_path = VIDEO_DIR / "ES2008a_video_analysis.json"
export_cols = ["meeting_id", "meeting_part", "start_time", "end_time", "description"]
with open(video_json_path, "w", encoding="utf-8") as fh:
    json.dump(
        video_df[export_cols].to_dict(orient="records"),
        fh, indent=2, ensure_ascii=False,
    )

print(f"\nSaved → {video_json_path}")

── Segment Duration Analysis ─────────────────────────────────────
  Segment count        : 10
  Avg segment duration : 104.5 s
  Total video covered  : 1041.1 s  (17.4 min)



,start_time,end_time,duration (s),description
0,00:00:00,00:01:18,78.26,"Rose Lindgren, the Project Manager, is present..."
1,00:01:18,00:02:14,56.00,The team is organizing their design project wo...
2,00:02:13,00:04:48,154.98,The meeting participants are introducing thems...
3,00:04:47,00:07:16,148.05,The group is engaging in an informal brainstor...
4,00:07:15,00:10:42,207.12,The team is discussing the financial projectio...
5,00:10:42,00:12:41,118.87,The group is discussing the design of a univer...
6,00:12:41,00:13:53,71.86,The team is brainstorming the design of a remo...
7,00:13:53,00:14:47,54.85,The team is discussing the design scope and fe...
8,00:14:47,00:16:28,100.37,The team is discussing the breakdown of indivi...
9,00:16:26,00:17:21,54.64,The team is concluding a meeting and discussin...



Saved → data\raw\video\ES2008a_video_analysis.json


In [21]:
_t0 = time.time()

seg_descriptions = video_df["description"].tolist()
seg_ids = [f"ES2008a_vseg_{i:02d}" for i in range(len(video_df))]

print(f"Embedding {len(seg_descriptions)} video segment descriptions …")
seg_embeddings = embed_model.encode(
    seg_descriptions, show_progress_bar=True, convert_to_numpy=True
)

for _name in ["video_segments"]:
    try:
        chroma_client.delete_collection(_name)
    except Exception:
        pass

video_col = chroma_client.create_collection(
    "video_segments",
    metadata={"hnsw:space": "cosine"},
)

video_col.add(
    ids=seg_ids,
    embeddings=seg_embeddings.tolist(),
    documents=seg_descriptions,
    metadatas=[
        {
            "meeting_part": row["meeting_part"],
            "start_time":   row["start_time"],
            "end_time":     row["end_time"],
            "start_sec":    float(row["start_sec"]),
            "end_sec":      float(row["end_sec"]),
            "duration_sec": float(row["duration_sec"]),
        }
        for _, row in video_df.iterrows()
    ],
)

step_times["Video Embedding"] = time.time() - _t0

print(f"\nChromaDB collections:")
print(f"  transcript_chunks : {transcript_col.count()} documents")
print(f"  slides_ocr        : {slides_col.count()} documents")
print(f"  video_segments    : {video_col.count()} documents")
print(f"  Embedding time    : {step_times['Video Embedding']:.2f} s")

Embedding 10 video segment descriptions …


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


ChromaDB collections:
  transcript_chunks : 27 documents
  slides_ocr        : 11 documents
  video_segments    : 10 documents
  Embedding time    : 0.19 s


In [22]:
def time_filtered_semantic_search(
    query: str,
    collection,
    start_hms: str,
    end_hms: str,
    similarity_threshold: float = 0.3,
    n_candidates: int = 20,
) -> pd.DataFrame:
    """
    Return segments overlapping [start_hms, end_hms] with similarity >= threshold.

    Threshold 0.3 is permissive: the time window already narrows candidates,
    so a low floor surfaces loosely related segments rather than missing them.
    """
    window_start = hhmmss_to_seconds(start_hms)
    window_end   = hhmmss_to_seconds(end_hms)

    q_emb = embed_model.encode([query], convert_to_numpy=True)
    res = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=min(n_candidates, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for seg_id, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0],
        res["metadatas"][0], res["distances"][0],
    ):
        sim = 1 - dist
        in_window = (
            meta["end_sec"] >= window_start and
            meta["start_sec"] <= window_end
        )
        if in_window and sim >= similarity_threshold:
            hits.append({
                "segment_id":  seg_id,
                "start_time":  meta["start_time"],
                "end_time":    meta["end_time"],
                "similarity":  round(sim, 4),
                "description": doc,
            })

    df = pd.DataFrame(hits)
    return (
        df.sort_values("similarity", ascending=False).reset_index(drop=True)
        if not df.empty else df
    )


# ── Example query: project goals in the first 5 minutes ───────────────────
QUERY_TIME    = "find segments about project goals in the first 5 minutes"
T_START       = "00:00:00"
T_END         = "00:05:00"
THRESH_SEARCH = 0.3

_t0 = time.time()
time_results = time_filtered_semantic_search(
    query=QUERY_TIME,
    collection=video_col,
    start_hms=T_START,
    end_hms=T_END,
    similarity_threshold=THRESH_SEARCH,
)
elapsed_ms = (time.time() - _t0) * 1000

print(f"Time + Semantic Search")
print(f"  Query     : {QUERY_TIME!r}")
print(f"  Window    : {T_START} → {T_END}")
print(f"  Threshold : similarity ≥ {THRESH_SEARCH}")
print(f"  Elapsed   : {elapsed_ms:.1f} ms")
print(f"  Matches   : {len(time_results)}\n")

if not time_results.empty:
    display(time_results.assign(description=time_results["description"].str[:100]))
else:
    print("No segments matched the time window and similarity threshold.")

Time + Semantic Search
  Query     : 'find segments about project goals in the first 5 minutes'
  Window    : 00:00:00 → 00:05:00
  Threshold : similarity ≥ 0.3
  Elapsed   : 18.6 ms
  Matches   : 2



,segment_id,start_time,end_time,similarity,description
0,ES2008a_vseg_00,00:00:00,00:01:18,0.3784,"Rose Lindgren, the Project Manager, is present..."
1,ES2008a_vseg_01,00:01:18,00:02:14,0.3288,The team is organizing their design project wo...


In [25]:
def aggregate_topic_time(
    query: str,
    collection,
    n_total: int,
    similarity_threshold: float = 0.7,
) -> dict:
    """
    Sum duration of segments with cosine similarity >= threshold.

    Threshold 0.7 is strict: counts only segments *predominantly* about the
    topic, not those where it appears incidentally.
    Returns total minutes and matched segment list.
    """
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    res = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=n_total,
        include=["documents", "metadatas", "distances"],
    )

    matching, total_sec = [], 0.0
    for seg_id, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0],
        res["metadatas"][0], res["distances"][0],
    ):
        sim = 1 - dist
        if sim >= similarity_threshold:
            dur = meta["duration_sec"]
            total_sec += dur
            matching.append({
                "segment_id":   seg_id,
                "start_time":   meta["start_time"],
                "end_time":     meta["end_time"],
                "duration_sec": round(dur, 1),
                "similarity":   round(sim, 4),
                "description":  doc,
            })

    return {
        "query":         query,
        "threshold":     similarity_threshold,
        "segment_count": len(matching),
        "total_minutes": round(total_sec / 60, 2),
        "segments":      matching,
    }


# ── Example aggregation ────────────────────────────────────────────────────
QUERY_AGG  = "remote control design and requirements"
THRESH_AGG = 0.6

_t0 = time.time()
agg = aggregate_topic_time(
    query=QUERY_AGG,
    collection=video_col,
    n_total=video_col.count(),
    similarity_threshold=THRESH_AGG,
)
elapsed_ms = (time.time() - _t0) * 1000

print(f"Time-Based Aggregation")
print(f"  Query          : {agg['query']!r}")
print(f"  Threshold      : similarity ≥ {agg['threshold']}")
print(f"  Elapsed        : {elapsed_ms:.1f} ms")
print(f"  Matching segs  : {agg['segment_count']}")
print(f"  Total time     : {agg['total_minutes']} min\n")

if agg["segments"]:
    agg_df = pd.DataFrame(agg["segments"])
    display(
        agg_df[["segment_id", "start_time", "end_time",
                "duration_sec", "similarity", "description"]]
        .assign(description=agg_df["description"].str[:80])
    )
else:
    print("No segments exceeded the similarity threshold.")

Time-Based Aggregation
  Query          : 'remote control design and requirements'
  Threshold      : similarity ≥ 0.6
  Elapsed        : 13.5 ms
  Matching segs  : 2
  Total time     : 2.11 min



,segment_id,start_time,end_time,duration_sec,similarity,description
0,ES2008a_vseg_07,00:13:53,00:14:47,54.9,0.6552,The team is discussing the design scope and fe...
1,ES2008a_vseg_06,00:12:41,00:13:53,71.9,0.6208,The team is brainstorming the design of a remo...
